# 对LLM基本代码使用用法的小结

## 基本环境的准备

In [1]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, PreTrainedModel, modeling_outputs
import torch
from loguru import logger
import os
import sys
from typing import cast

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True)

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 基础总结1：使用 GPT-2 完成下一个 Token 预测与句子生成

In [2]:
class GPT2GenerateToken:
    def __init__(self, tokenizer: GPT2Tokenizer, model: PreTrainedModel) -> None:
        self._tokenizer: GPT2Tokenizer = tokenizer
        self._model: PreTrainedModel = model
        self._top_k: int = 3  # 默认topk设置为3

    def generate_next_token(self, prompt: str) -> None:
        """
        模型生成基本的下一个token
        :param prompt: 提示词字符串
        :return: None
        """
        logger.info(f"Step1: 从提示词字符串->对应的token id列表->对应的张量类形式->分词后的tokens列表")
        logger.info(f"prompt : {prompt}")
        input_ids: list[int] = self._tokenizer.encode(prompt)
        logger.info(f"对应的token ids ：{input_ids}")
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)
        logger.info(f"对应的张量类的表示：{token_ids}")
        tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in input_ids]
        logger.info(f"对应的分开的词的形式：{tokens}")

        with torch.no_grad():
            output: modeling_outputs.CausalLMOutputWithCrossAttentions = self._model(token_ids)
            logits: torch.Tensor = cast(torch.Tensor, output.logits)
            logger.info(
                f"logits张量的shape为：{logits.shape}，其中的第一个维度是batch，第二个维度是token数，第三个维度是词表数")
            next_token_logits: torch.Tensor = logits[0, -1, :]
            logger.info(f"用于打分的logits维度是{next_token_logits.shape}\n值为{next_token_logits}")
            next_token_probabilities: torch.Tensor = torch.softmax(next_token_logits, dim=0)
            logger.info(f"经过softmax后的概率shape：{next_token_probabilities.shape}，概率是{next_token_probabilities}")
            top_k_probabilities, top_k_index = torch.topk(next_token_probabilities, self._top_k)
            top_k_tokens: list[str] = [cast(str, self._tokenizer.decode(token_id)) for token_id in top_k_index]
            for i, (probability, index, token) in enumerate(zip(top_k_probabilities, top_k_index, top_k_tokens)):
                logger.info(f"Top {i} : probability = {probability}, index = {index}, token = {token}")

    def generate_sentence(self, prompt: str, max_token_num: int = 50) -> str:
        """
        模型进行基本的续写
        :param max_token_num: 最大句子长度限制
        :param prompt: 提示词字符串
        :return: 续写的字符串
        """
        # process basic input
        input_ids: list[int] = self._tokenizer.encode(prompt)
        token_ids: torch.Tensor = torch.tensor([input_ids], dtype=torch.long)

        # logits and probabilities, use non-greedy settings
        should_generate_end: bool = False
        while not should_generate_end:
            with torch.no_grad():
                logits: torch.Tensor = cast(torch.Tensor, self._model(token_ids).logits)[0, -1, :]
                probabilities: torch.Tensor = torch.softmax(logits, dim=0)
                next_token_id: int = cast(int, torch.argmax(probabilities).item())
                next_token: str = cast(str, self._tokenizer.decode(next_token_id))

                prompt += next_token
                token_ids = torch.cat([token_ids, torch.tensor([[next_token_id]], dtype=torch.long)], dim=1)

                if next_token in [".", "?", "!"] or token_ids.shape[1] >= max_token_num:
                    should_generate_end = True
        return prompt

展示GPT-2生成下一个token的详细过程

In [3]:
gpt2_generate_token = GPT2GenerateToken(gpt2_tokenizer, gpt2_model)
gpt2_generate_token.generate_next_token("Thank you very")

使用GPT-2进行句子续写，使用greedy settings

In [4]:
answer: str = gpt2_generate_token.generate_sentence("The meaning of life is", max_token_num=50)
logger.info(answer)

2026-08-08 16:19:51.147 | INFO     | __main__:<module>:2 - The meaning of life is not the same as the meaning of death.


## 基础总结2：GPT-2 对下一个 Token 的打分与采样过程

## 基础总结3：从文本到 Token、Embedding 与上下文 Feature

## 基础总结4：WTE 与 WPE 如何共同构成 Transformer 的输入